# Lesson 0. Fundamentals: the problem, cross-validation, and a model by hand

You are entering the **RSNA Knee MRI** Kaggle competition. Each knee study has
labels for **12 targets** (ACL and MCL tears, meniscus tears, osteoarthritis in
three compartments, effusion, synovitis, Baker's cyst, contusion, and fracture).
Your task is to predict each target from the MRI.

Before you download a large model, you need three tools. They decide whether a
score you report is real:

1. **Cross-validation**, to estimate performance without a false result.
2. **A classifier head**, which turns features into a prediction.
3. **The shuffle control**, the cheapest check that the model learned something.

This lesson builds all three by hand on a simple feature, so you can see how each
one works. Later lessons put in real encoders. The three tools do not change.

**Anatomy orientation.** If the target names are new to you, open a knee atlas and
find the ACL, the menisci, and the joint effusion space before you model them:

- OpenAnatomy knee atlas (labeled 3-D structures): <https://www.openanatomy.org/>
- Open Knee(s), real knee MRI with segmentations: <https://simtk.org/projects/openknee>
- Radiopaedia, normal knee MRI: <https://radiopaedia.org/articles/knee-joint-1>

More atlases, encoders, and generation methods are in the course reading page,
`references/approaches_and_reading.md`.

**Where you are: Lesson 0 of 3.** The order is L0 fundamentals, then L1 naive
baseline, then L2 encoder comparison. In this lesson you build the three tools that
tell you whether a score is real. There are no pretrained models yet. That is
Lesson 1.

How to read this notebook:

- **Step** headers tell you what you do now.
- **Experiment** cells are optional. Change one setting, run the cell again, and
  look at the result. Skip them for the fast path. Run them to learn more.
- **Go deeper** links point to the theory.

## Step 1: Get the data without a full download

The competition images are **about 570 GB of DICOMs**. You will not download that,
and you do not need to. On a Kaggle kernel the competition is **mounted** at
`/kaggle/input/...`, so the files are on disk and are not copied to you. The method
is to read a bounded sample: the first `N_STUDIES` studies, a few slices each.

**To run this notebook,** join the competition, accept its rules, then use
*Add Input* and select this competition. If a cell does not find the data, correct
`DATA_ROOT`.

In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# config: read a bounded sample, never the full ~570 GB ----------------------
DATA_ROOT = os.environ.get("RSNA_DATA_ROOT",
    "/kaggle/input/rsna-knee-abnormality-detection")  # competition mount
N_STUDIES = 300     # read only this many studies' DICOMs, not the full dataset
K_SLICES  = 5       # central slices per study
SEED      = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def _find_file(root, names):
    for n in names:
        p = os.path.join(root, n)
        if os.path.isfile(p):
            return p
    for n in names:
        hits = sorted(glob.glob(os.path.join(root, "**", n), recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"none of {names} under {root}")

def _find_dir(root, name):
    p = os.path.join(root, name)
    if os.path.isdir(p):
        return p
    hits = sorted(d for d in glob.glob(os.path.join(root, "**", name), recursive=True)
                  if os.path.isdir(d))
    if not hits:
        raise FileNotFoundError(f"dir {name} not found under {root}")
    return hits[0]

TRAIN_CSV        = _find_file(DATA_ROOT, ["train.csv", "metadata/train.csv"])
SERIES_CSV       = _find_file(DATA_ROOT, ["train_series.csv", "metadata/train_series.csv"])
TRAIN_SERIES_DIR = _find_dir(DATA_ROOT, "train_series")
print("labels :", TRAIN_CSV)
print("series :", SERIES_CSV)
print("dicoms :", TRAIN_SERIES_DIR)

In [ ]:
import pydicom

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def list_studies(train_series_dir, n=None):
    studies = sorted(d for d in os.listdir(train_series_dir)
                     if os.path.isdir(os.path.join(train_series_dir, d)))
    return studies[:n] if n else studies

def first_sagittal_series(series_df, study, train_series_dir):
    study_dir = os.path.join(train_series_dir, study)
    rows = series_df[(series_df["StudyInstanceUID"] == study) &
                     (series_df["Anatomical_Plane"].astype(str).str.lower() == "sagittal")]
    for sid in rows["SeriesInstanceUID"].astype(str):
        sdir = os.path.join(study_dir, sid)
        if os.path.isdir(sdir) and glob.glob(os.path.join(sdir, "*.dcm")):
            return sdir
    for sdir in sorted(glob.glob(os.path.join(study_dir, "*"))):
        if glob.glob(os.path.join(sdir, "*.dcm")):
            return sdir
    return None

def _decode(ds):
    px = np.asarray(ds.pixel_array, dtype=np.float32)
    px = px * float(getattr(ds, "RescaleSlope", 1.0)) + float(getattr(ds, "RescaleIntercept", 0.0))
    if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")) == "MONOCHROME1":
        px = float(px.max() + px.min()) - px
    return px

def load_central_slices(series_dir, k=K_SLICES):
    recs = []
    for i, p in enumerate(sorted(glob.glob(os.path.join(series_dir, "*.dcm")))):
        try:
            ds = pydicom.dcmread(p)
        except Exception:
            continue
        try:
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            coord = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            coord = float(i)
        try:
            px = _decode(ds)
        except Exception:
            continue
        if px.ndim == 2:
            recs.append((coord, px))
    if not recs:
        return []
    recs.sort(key=lambda r: r[0])
    lo = max(0, len(recs) // 2 - k // 2)
    return [px for _, px in recs[lo:lo + k]]

def _norm01(img):
    finite = img[np.isfinite(img)]
    if finite.size == 0:
        return np.zeros_like(img, dtype=np.float32)
    lo, hi = np.percentile(finite, [1, 99])
    if hi <= lo:
        return np.zeros_like(img, dtype=np.float32)
    clean = np.nan_to_num(img, nan=float(lo), posinf=float(hi), neginf=float(lo))
    return np.clip((clean - lo) / (hi - lo), 0, 1).astype(np.float32)

def prep_batch(slices, size=224):
    frames = []
    for s in slices:
        t = torch.from_numpy(_norm01(s))[None, None]
        t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
        frames.append(t[0, 0])
    x = torch.stack(frames)[:, None].repeat(1, 3, 1, 1)
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (x - mean) / std

## Step 2: Look at the labels first

Class balance decides which metric is honest. Print how often each target is
positive. Several targets are rare. For that reason you score with **AUC**, a
ranking metric, not accuracy. An "always negative" model can get high accuracy.

In [ ]:
labels = pd.read_csv(TRAIN_CSV)
series = pd.read_csv(SERIES_CSV)
findings = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
            "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
prev = (labels[findings].mean().sort_values(ascending=False) * 100).round(1)
print("positive rate (%) across", len(labels), "studies:")
print(prev.to_string())

## Step 3: Cross-validation, from scratch

You never score a model on data it trained on, because it has seen the answers.
Split the studies into **k folds**. Train on k-1 folds, predict the held-out fold,
and repeat. Each study then gets one **out-of-fold** prediction from a model that
did not see it.

Two rules that people get wrong on medical data:

- **Group by patient or study.** If two slices of the same knee are on opposite
  sides of a split, the model predicts a knee it already learned. This notebook uses
  one row per study, so a study is never split.
- **Stratify.** With rare targets, a random split can put all the positives in one
  fold. Stratified folds keep the positive rate about equal across folds.

`cv_auc` below is the full idea in about ten lines.

Go deeper: [scikit-learn cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html),
[ROC and AUC](https://scikit-learn.org/stable/auto_examples/model_selection/plot_roc.html).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

def cv_auc(X, y, seed=SEED, n_splits=5, shuffle_labels=False):
    """Out-of-fold ROC-AUC. Each study is one row, so folds never split a study."""
    y = np.asarray(y).astype(int)
    if shuffle_labels:
        y = y[np.random.default_rng(seed).permutation(len(y))]
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    for tr, va in skf.split(X, y):
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=1000).fit(sc.transform(X[tr]), y[tr])
        oof[va] = clf.predict_proba(sc.transform(X[va]))[:, 1]
    return float(roc_auc_score(y, oof))

def mean_auc(X, y, shuffle_labels=False, seeds=range(10)):
    """Average AUC over several CV seeds -- stable when N is small and noisy."""
    return float(np.mean([cv_auc(X, y, seed=s, shuffle_labels=shuffle_labels)
                          for s in seeds]))

## Step 4: A feature by hand, and why a simple one fails

Build the simplest possible feature: four intensity statistics of the central
slices. Then fit the head and cross-validate. Expect an AUC **near 0.5**, because
the average brightness of a knee does not show whether the ACL is torn. The failure
is the lesson. It tells you the feature, not the pipeline, is what is missing.
Lessons 1 and 2 fix the feature.

In [ ]:
def intensity_feature(slices):
    x = np.stack([_norm01(s) for s in slices])
    return np.array([x.mean(), x.std(),
                     np.percentile(x, 90), np.percentile(x, 10)], dtype=np.float32)

studies = list_studies(TRAIN_SERIES_DIR, n=min(N_STUDIES, 150))  # keep L0 quick
lab = labels.set_index("StudyInstanceUID")
X, kept = [], []
for s in studies:
    sd = first_sagittal_series(series, s, TRAIN_SERIES_DIR)
    if sd is None:
        continue
    sl = load_central_slices(sd)
    if not sl:
        continue
    X.append(intensity_feature(sl)); kept.append(s)
X = np.stack(X)
y = lab.loc[kept, "Effusion"].to_numpy()
print(f"{len(kept)} studies | effusion +{int(y.sum())}/-{int(len(y)-y.sum())}")
print("intensity-feature AUC (effusion), 10-seed mean:", round(mean_auc(X, y), 3))

## Step 5: The classifier head, logistic regression then a tiny MLP by hand

`cv_auc` used the scikit-learn `LogisticRegression`. It finds one weight per feature,
so that a weighted sum, mapped to the 0 to 1 range, matches the labels. That is a
one-layer network. To make it clear, the next cell writes the same idea as an
explicit PyTorch training loop (forward, loss, backward, step) on the same features.

In [ ]:
Xt = torch.tensor(StandardScaler().fit_transform(X), dtype=torch.float32)
yt = torch.tensor(y, dtype=torch.float32)

torch.manual_seed(SEED)
mlp = torch.nn.Sequential(torch.nn.Linear(X.shape[1], 8), torch.nn.ReLU(),
                          torch.nn.Linear(8, 1))
opt = torch.optim.Adam(mlp.parameters(), lr=0.05)
lossf = torch.nn.BCEWithLogitsLoss()
for epoch in range(200):
    opt.zero_grad()
    loss = lossf(mlp(Xt).squeeze(1), yt)
    loss.backward()
    opt.step()
print("final training loss:", round(loss.item(), 3),
      "  (in-sample only, not a real score; see the shuffle control next)")

## Step 6: The shuffle control, did it learn anything?

Permute the labels at random, so the features and labels have **no real
relationship**. Then run the same CV. A model that learned real signal scores
clearly above its shuffled version. A model that only overfit scores about the same,
shuffled or not.

On the intensity feature, the real score and the shuffle score both sit near 0.5,
because there was no signal to find. Keep this control. In Lesson 1 you see the two
scores separate. In the instructor examples you see a trained model that the shuffle
score beats.

In [ ]:
print("intensity feature, real    AUC (10-seed mean):", round(mean_auc(X, y), 3))
print("intensity feature, shuffle AUC (10-seed mean):", round(mean_auc(X, y, shuffle_labels=True), 3))
print("(single seeds vary widely at N=58; the average is why both sit near 0.5)")

## Experiment: what happens if you change the seed?

The average above hides one detail. Run one seed at a time, and the shuffle AUC
varies across a wide range, because 58 studies split six ways leave only a few
positives per fold. Predict the range before you run the cell, then look. This
variation is the reason you average. It is the same small-sample noise that decides
the encoder comparison in Lesson 2.

In [ ]:
for seed in range(6):
    print(f"seed={seed}  shuffle AUC = {cv_auc(X, y, seed=seed, shuffle_labels=True):.3f}")

## Recap

You built cross-validation, a classifier head, and the shuffle control on a feature
with no signal, so each one is easy to see. Next, in Lesson 1, you replace the
simple feature with a pretrained ImageNet encoder and see the shuffle gap open.